In [1]:
train = "/Users/DYNAMIC-COMPUTER/Desktop/PJ-2 Gene Expression Classifier/data_set_ALL_AML_train.csv"
test = "/Users/DYNAMIC-COMPUTER/Desktop/PJ-2 Gene Expression Classifier/data_set_ALL_AML_independent.csv"
class_labels = "/Users/DYNAMIC-COMPUTER/Desktop/PJ-2 Gene Expression Classifier/actual.csv"

In [2]:
import pandas as pd
train_df = pd.read_csv(train, encoding='utf-8')
test_df = pd.read_csv(test, encoding='utf-8')
class_labels_df = pd.read_csv(class_labels, encoding='utf-8')

In [3]:
train_updated = train_df.set_index(train_df.columns[0])
train_updated = train_updated.T
train_updated.reset_index(inplace=True)
train_updated.rename(columns={'index': 'Sample'}, inplace=True)

In [4]:
class_labels_df.rename(columns={"patient": "Sample", "cancer": "Class"}, inplace=True)

In [5]:
train_updated["Sample"] = train_updated["Sample"].astype(str)
class_labels_df["Sample"] = class_labels_df["Sample"].astype(str)

In [6]:
train_final = pd.merge(train_updated, class_labels_df, on='Sample')

As we know, we removed AFFX (Quality Control Probes) for better ML accuracy.

In [7]:
train_updated_without_affx = train_final.loc[:, ~train_final.columns.str.startswith('AFFX-')]

TRAIN TEST SPLIT

In [8]:
X = train_updated_without_affx.drop(columns=['Sample', 'Class'])
y = train_updated_without_affx['Class']

In [9]:
from sklearn.model_selection import train_test_split
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

PREPROCESSING

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

preprocessing = Pipeline([
    ("variance_threshold", VarianceThreshold(threshold=0.01)),
    ("standard_scaler", StandardScaler()),
    ("pca", PCA(n_components=0.95))
])

Logistic Regression

In [11]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

log_reg_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("log_reg", LogisticRegression(max_iter=500))
])

skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

score = cross_val_score(log_reg_pipeline, X, y, cv=skfold)

In [12]:
score

array([1.  , 1.  , 0.75, 1.  , 1.  ])

In [13]:
score.mean()

np.float64(0.95)

In [14]:
from sklearn.ensemble import RandomForestClassifier

rnd_clf_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("rnd_clf", RandomForestClassifier(n_estimators=500, random_state=42))
])

score = cross_val_score(rnd_clf_pipeline, X, y, cv=skfold)

In [15]:
score

array([0.75      , 0.75      , 0.75      , 0.85714286, 0.85714286])

In [16]:
score.mean()

np.float64(0.7928571428571429)

XGB CLASSIFIER

In [17]:
from xgboost import XGBClassifier

xgb_pipeline = Pipeline([
    ('preprocessing', preprocessing),
    ('xgb_clf', XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        random_state=42,
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8
    ))
])
y_encoded = y.map({'AML': 0, 'ALL': 1})
score = cross_val_score(xgb_pipeline, X, y_encoded, cv=skfold)

In [18]:
score

array([0.75 , 0.75 , 0.875, 1.   , 1.   ])

In [19]:
score.mean()

np.float64(0.875)

DECISION TREE CLASSIFIER

In [20]:
from sklearn.tree import DecisionTreeClassifier
dtree_pipeline = Pipeline([
    ('preprocessing', preprocessing),
    ('dtree', DecisionTreeClassifier(max_depth=2, random_state=42))
])
score = cross_val_score(dtree_pipeline, X, y, cv=skfold)

In [21]:
score

array([0.75      , 0.75      , 0.75      , 0.85714286, 1.        ])

In [22]:
score.mean()

np.float64(0.8214285714285715)

Logistic Regression has shown the most accruacy at .95